# Notebook 03: Vocabulary + Pairs (MARS)

**Purpose:** Build item vocabulary + (prefix → next item) pairs for next-item prediction.

**Inputs:**
- `data/interim/mars.duckdb` → view `mars_events_sessionized`

**Outputs:**
- `data/processed/mars/vocab/item2id.json`
- `data/processed/mars/vocab/id2item.json`
- `data/processed/mars/pairs/pairs.parquet`
- `data/interim/mars.duckdb` → view `mars_pairs`
- `reports/03_vocab_pairs_mars/<run_tag>/report.json`

In [1]:
# [CELL 03-00] Bootstrap

import json
import time
import uuid
import hashlib
from pathlib import Path
from datetime import datetime
from typing import Any

import numpy as np
import pandas as pd

def find_repo_root(start):
    for p in [Path(start).resolve(), *Path(start).resolve().parents]:
        if (p / 'PROJECT_STATE.md').exists(): return p
    raise RuntimeError('PROJECT_STATE.md not found')

REPO_ROOT = find_repo_root(Path.cwd())
PATHS = {
    'META_REGISTRY':  REPO_ROOT / 'meta.json',
    'DATA_INTERIM':   REPO_ROOT / 'data' / 'interim',
    'DATA_PROCESSED': REPO_ROOT / 'data' / 'processed',
    'REPORTS':        REPO_ROOT / 'reports',
}

def cell_start(cid, title, **kw):
    t = time.time()
    print(f'\n[{cid}] {title}')
    print(f'[{cid}] start={datetime.now().isoformat(timespec="seconds")}')
    for k, v in kw.items(): print(f'[{cid}] {k}={v}')
    return t

def cell_end(cid, t0, **kw):
    for k, v in kw.items(): print(f'[{cid}] {k}={v}')
    print(f'[{cid}] elapsed={time.time()-t0:.2f}s  done')

def write_json_atomic(path, obj, indent=2):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + f'.tmp_{uuid.uuid4().hex}')
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=indent), encoding='utf-8')
    tmp.replace(path)

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(1 << 20): h.update(chunk)
    return h.hexdigest()

print(f'[CELL 03-00] REPO_ROOT={REPO_ROOT}  done')

[CELL 03-00] REPO_ROOT=/home/user/jamalla/anonymous-users-mooc-session-meta  done


In [2]:
# [CELL 03-01] Seed + run config

import random

GLOBAL_SEED = 20260107
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

NOTEBOOK_NAME = '03_vocab_pairs_mars'
DATASET       = 'mars'

RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_ID  = uuid.uuid4().hex

OUT_DIR       = PATHS['REPORTS'] / NOTEBOOK_NAME / RUN_TAG
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH   = OUT_DIR / 'report.json'
CONFIG_PATH   = OUT_DIR / 'config.json'
MANIFEST_PATH = OUT_DIR / 'manifest.json'

DUCKDB_PATH  = PATHS['DATA_INTERIM'] / 'mars.duckdb'
EVENTS_VIEW  = 'mars_events_sessionized'

VOCAB_DIR = PATHS['DATA_PROCESSED'] / 'mars' / 'vocab'
PAIRS_DIR = PATHS['DATA_PROCESSED'] / 'mars' / 'pairs'
VOCAB_DIR.mkdir(parents=True, exist_ok=True)
PAIRS_DIR.mkdir(parents=True, exist_ok=True)

OUT_ITEM2ID = VOCAB_DIR / 'item2id.json'
OUT_ID2ITEM = VOCAB_DIR / 'id2item.json'
OUT_PAIRS   = PAIRS_DIR / 'pairs.parquet'

CFG = {
    'notebook': NOTEBOOK_NAME, 'dataset': DATASET, 'seed': GLOBAL_SEED,
    'pairs': {'min_prefix_len': 1, 'deduplicate_consecutive': True},
}
write_json_atomic(CONFIG_PATH, CFG)

for path, obj in [
    (REPORT_PATH, {'run_id': RUN_ID, 'notebook': NOTEBOOK_NAME, 'run_tag': RUN_TAG,
                   'created_at': datetime.now().isoformat(timespec='seconds'),
                   'metrics': {}, 'key_findings': [], 'sanity_samples': {},
                   'data_fingerprints': {}, 'notes': []}),
    (MANIFEST_PATH, {'run_id': RUN_ID, 'notebook': NOTEBOOK_NAME, 'run_tag': RUN_TAG, 'artifacts': []}),
]:
    write_json_atomic(path, obj)

META = PATHS['META_REGISTRY']
if not META.exists(): write_json_atomic(META, {'schema_version': 1, 'runs': []})
m = read_json(META)
m['runs'].append({'run_id': RUN_ID, 'notebook': NOTEBOOK_NAME, 'run_tag': RUN_TAG, 'out_dir': str(OUT_DIR)})
write_json_atomic(META, m)

print(f'[CELL 03-01] RUN_TAG={RUN_TAG}  DUCKDB={DUCKDB_PATH}')
print('[CELL 03-01] done')

[CELL 03-01] RUN_TAG=20260409_180743  DUCKDB=/home/user/jamalla/anonymous-users-mooc-session-meta/data/interim/mars.duckdb
[CELL 03-01] done


In [3]:
# [CELL 03-02] Load sessionized events from DuckDB

import duckdb

t0 = cell_start('CELL 03-02', 'Load events from DuckDB', view=EVENTS_VIEW)

if not DUCKDB_PATH.exists():
    raise RuntimeError(f'Missing {DUCKDB_PATH}. Run 02_sessionize_mars.ipynb first.')

con = duckdb.connect(str(DUCKDB_PATH), read_only=True)
events = con.execute(f"""
    SELECT user_id, item_id, session_id, ts_epoch
    FROM {EVENTS_VIEW}
    ORDER BY user_id, session_id, ts_epoch
""").fetchdf()
con.close()

print(f'[CELL 03-02] events shape: {events.shape}')
print(f'[CELL 03-02] Columns: {list(events.columns)}')
print(f'[CELL 03-02] n_users: {events["user_id"].nunique():,}')
print(f'[CELL 03-02] n_items: {events["item_id"].nunique():,}')
print(f'[CELL 03-02] n_sessions: {events["session_id"].nunique():,}')

cell_end('CELL 03-02', t0, n_events=int(len(events)))


[CELL 03-02] Load events from DuckDB
[CELL 03-02] start=2026-04-09T18:07:44
[CELL 03-02] view=mars_events_sessionized
[CELL 03-02] events shape: (2894, 4)
[CELL 03-02] Columns: ['user_id', 'item_id', 'session_id', 'ts_epoch']
[CELL 03-02] n_users: 378
[CELL 03-02] n_items: 745
[CELL 03-02] n_sessions: 561
[CELL 03-02] n_events=2894
[CELL 03-02] elapsed=0.04s  done


In [4]:
# [CELL 03-03] Build item vocabulary (sorted → deterministic 0-indexed)

t0 = cell_start('CELL 03-03', 'Build item vocabulary')

unique_items = sorted(events['item_id'].unique())
item2id = {str(item): i for i, item in enumerate(unique_items)}
id2item = {i: str(item) for item, i in item2id.items()}
n_items = len(item2id)

write_json_atomic(OUT_ITEM2ID, item2id)
write_json_atomic(OUT_ID2ITEM, id2item)

print(f'[CELL 03-03] n_items={n_items}')
print(f'[CELL 03-03] First 5 items: {unique_items[:5]}')
print(f'[CELL 03-03] Saved: {OUT_ITEM2ID}')
print(f'[CELL 03-03] Saved: {OUT_ID2ITEM}')

cell_end('CELL 03-03', t0, n_items=n_items)


[CELL 03-03] Build item vocabulary
[CELL 03-03] start=2026-04-09T18:07:44
[CELL 03-03] n_items=745
[CELL 03-03] First 5 items: [510, 511, 512, 513, 514]
[CELL 03-03] Saved: /home/user/jamalla/anonymous-users-mooc-session-meta/data/processed/mars/vocab/item2id.json
[CELL 03-03] Saved: /home/user/jamalla/anonymous-users-mooc-session-meta/data/processed/mars/vocab/id2item.json
[CELL 03-03] n_items=745
[CELL 03-03] elapsed=0.00s  done


In [5]:
# [CELL 03-04] Map item_id → item_idx; build session sequences (dedup consecutive)

t0 = cell_start('CELL 03-04', 'Map IDs + build session sequences')

# Map item_id to integer index
events['item_idx'] = events['item_id'].map(lambda x: item2id[str(x)])
assert events['item_idx'].isna().sum() == 0, 'Unmapped items found'

DEDUPE = CFG['pairs']['deduplicate_consecutive']

def dedupe_consecutive(items, tss):
    """Remove consecutive duplicate items, keeping first occurrence timestamp."""
    di, dt = [items[0]], [tss[0]]
    for i in range(1, len(items)):
        if items[i] != di[-1]:
            di.append(items[i])
            dt.append(tss[i])
    return di, dt

session_seqs = []
for (uid, sid), g in events.groupby(['user_id', 'session_id']):
    g = g.sort_values('ts_epoch')
    items = g['item_idx'].tolist()
    tss   = g['ts_epoch'].tolist()
    if DEDUPE:
        items, tss = dedupe_consecutive(items, tss)
    session_seqs.append({
        'user_id':    uid,
        'session_id': sid,
        'item_seq':   items,
        'ts_seq':     tss,
    })

print(f'[CELL 03-04] {len(session_seqs):,} session sequences')
seq_lens = [len(s['item_seq']) for s in session_seqs]
print(f'[CELL 03-04] Sequence length: min={min(seq_lens)}  p50={np.percentile(seq_lens, 50):.0f}  p90={np.percentile(seq_lens, 90):.0f}  max={max(seq_lens)}')

cell_end('CELL 03-04', t0, n_sessions=len(session_seqs))


[CELL 03-04] Map IDs + build session sequences
[CELL 03-04] start=2026-04-09T18:07:44
[CELL 03-04] 561 session sequences
[CELL 03-04] Sequence length: min=2  p50=3  p90=11  max=50
[CELL 03-04] n_sessions=561
[CELL 03-04] elapsed=0.10s  done


In [6]:
# [CELL 03-05] Create prefix→label pairs (chronological, no leakage)

t0 = cell_start('CELL 03-05', 'Create prefix→label pairs')

MIN_PFX = CFG['pairs']['min_prefix_len']
pairs = []
pid = 0

for sess in session_seqs:
    items = sess['item_seq']
    tss   = sess['ts_seq']
    if len(items) < MIN_PFX + 1:
        continue
    for t in range(MIN_PFX, len(items)):
        pfx_ts = tss[:t]
        lbl_ts = tss[t]
        # Strict chronological: all prefix timestamps must precede label timestamp
        if max(pfx_ts) >= lbl_ts:
            continue
        pairs.append({
            'pair_id':        pid,
            'user_id':        sess['user_id'],
            'session_id':     sess['session_id'],
            'prefix':         items[:t],
            'label':          int(items[t]),
            'label_ts_epoch': int(lbl_ts),
            'prefix_len':     t,
        })
        pid += 1

pairs_df = pd.DataFrame(pairs)
print(f'[CELL 03-05] {len(pairs_df):,} pairs created')
print(f'[CELL 03-05] prefix_len: p50={pairs_df["prefix_len"].quantile(.5):.0f}  p90={pairs_df["prefix_len"].quantile(.9):.0f}  max={pairs_df["prefix_len"].max()}')
print(f'[CELL 03-05] Columns: {list(pairs_df.columns)}')

cell_end('CELL 03-05', t0, n_pairs=int(len(pairs_df)))


[CELL 03-05] Create prefix→label pairs
[CELL 03-05] start=2026-04-09T18:07:44
[CELL 03-05] 2,333 pairs created
[CELL 03-05] prefix_len: p50=4  p90=18  max=49
[CELL 03-05] Columns: ['pair_id', 'user_id', 'session_id', 'prefix', 'label', 'label_ts_epoch', 'prefix_len']
[CELL 03-05] n_pairs=2333
[CELL 03-05] elapsed=0.02s  done


In [7]:
# [CELL 03-06] Validate: labels and prefix items all < n_items

t0 = cell_start('CELL 03-06', 'Validation checks')

assert (pairs_df['label'] < n_items).all(), f'Label out of vocab range [0, {n_items})'
assert (pairs_df['label'] >= 0).all(), 'Negative label found'

all_pfx_items = [x for pfx in pairs_df['prefix'] for x in pfx]
if all_pfx_items:
    assert max(all_pfx_items) < n_items, f'Prefix item exceeds vocab size {n_items}'
    assert min(all_pfx_items) >= 0, 'Negative prefix item found'

assert (pairs_df['prefix_len'] >= MIN_PFX).all(), f'Short prefix (< {MIN_PFX}) found'

print(f'[CELL 03-06] All labels in [0, {n_items})')
print(f'[CELL 03-06] All prefix items in [0, {n_items})')
print(f'[CELL 03-06] All prefix lengths >= {MIN_PFX}')

user_counts = pairs_df.groupby('user_id').size()
print(f'[CELL 03-06] Users with pairs: {len(user_counts):,}')
print(f'[CELL 03-06] Pairs per user: min={user_counts.min()}  p50={user_counts.quantile(.5):.0f}  p90={user_counts.quantile(.9):.0f}  max={user_counts.max()}')

# Session coverage
sess_counts = pairs_df.groupby('session_id').size()
print(f'[CELL 03-06] Sessions with pairs: {len(sess_counts):,}')

print('[CELL 03-06] All validation checks passed.')
cell_end('CELL 03-06', t0)


[CELL 03-06] Validation checks
[CELL 03-06] start=2026-04-09T18:07:44
[CELL 03-06] All labels in [0, 745)
[CELL 03-06] All prefix items in [0, 745)
[CELL 03-06] All prefix lengths >= 1
[CELL 03-06] Users with pairs: 378
[CELL 03-06] Pairs per user: min=1  p50=2  p90=13  max=123
[CELL 03-06] Sessions with pairs: 561
[CELL 03-06] All validation checks passed.
[CELL 03-06] elapsed=0.00s  done


In [9]:
# [CELL 03-07] Save pairs.parquet + register DuckDB view + save report

t0 = cell_start('CELL 03-07', 'Save pairs.parquet + register DuckDB view')

pairs_df.to_parquet(OUT_PAIRS, index=False, compression='zstd')
pairs_bytes = int(OUT_PAIRS.stat().st_size)
pairs_sha   = sha256_file(OUT_PAIRS)

print(f'[CELL 03-07] Saved {OUT_PAIRS} ({pairs_bytes / (1 << 20):.1f} MB)')
print(f'[CELL 03-07] SHA256: {pairs_sha}')

# Register DuckDB view
def esc(p): return str(p).replace("'", "''")

con = duckdb.connect(str(DUCKDB_PATH), read_only=False)
con.execute('DROP VIEW IF EXISTS mars_pairs;')
con.execute(f"CREATE VIEW mars_pairs AS SELECT * FROM read_parquet('{esc(OUT_PAIRS)}')")
n_view = int(con.execute('SELECT COUNT(*) FROM mars_pairs').fetchone()[0])
con.close()

print(f'[CELL 03-07] DuckDB view mars_pairs: {n_view:,} rows')

# Update report
r = read_json(REPORT_PATH)
r['metrics'] = {
    'n_items':  n_items,
    'n_pairs':  int(len(pairs_df)),
    'n_users':  int(len(user_counts)),
    'n_sessions_with_pairs': int(len(sess_counts)),
}
r['data_fingerprints']['pairs'] = {'path': str(OUT_PAIRS), 'bytes': pairs_bytes, 'sha256': pairs_sha}
r['data_fingerprints']['item2id'] = {'path': str(OUT_ITEM2ID)}
r['data_fingerprints']['id2item'] = {'path': str(OUT_ID2ITEM)}
r['sanity_samples']['pairs_head3'] = pairs_df.head(3).to_dict(orient='records')
r['notes'].append(f'Vocabulary: {n_items} unique items (0-indexed). Consecutive duplicates removed within sessions.')
write_json_atomic(REPORT_PATH, r)

manifest = read_json(MANIFEST_PATH)
for path in [OUT_ITEM2ID, OUT_ID2ITEM, OUT_PAIRS]:
    try: sha = sha256_file(path)
    except Exception as e: sha = f'ERROR: {e}'
    manifest['artifacts'].append({'path': str(path), 'bytes': int(path.stat().st_size), 'sha256': sha})
write_json_atomic(MANIFEST_PATH, manifest)

print(f'\n[CELL 03-07] Report: {REPORT_PATH}')
print(f'[CELL 03-07] Summary: n_items={n_items}  n_pairs={len(pairs_df):,}  n_users={len(user_counts):,}')
cell_end('CELL 03-07', t0, n_pairs=int(len(pairs_df)))


[CELL 03-07] Save pairs.parquet + register DuckDB view
[CELL 03-07] start=2026-04-09T18:08:35
[CELL 03-07] Saved /home/user/jamalla/anonymous-users-mooc-session-meta/data/processed/mars/pairs/pairs.parquet (0.0 MB)
[CELL 03-07] SHA256: 3856bed3fc43eec1ea5a6e0790039acca6fbb7a052de639a2ac2918a620a67bc
[CELL 03-07] DuckDB view mars_pairs: 2,333 rows

[CELL 03-07] Report: /home/user/jamalla/anonymous-users-mooc-session-meta/reports/03_vocab_pairs_mars/20260409_180743/report.json
[CELL 03-07] Summary: n_items=745  n_pairs=2,333  n_users=378
[CELL 03-07] n_pairs=2333
[CELL 03-07] elapsed=0.37s  done


## Notebook 03 Complete

**Outputs:**
- `data/processed/mars/vocab/item2id.json` — item → integer index mapping
- `data/processed/mars/vocab/id2item.json` — integer index → item mapping
- `data/processed/mars/pairs/pairs.parquet` — (prefix → label) pairs with session_id
- `data/interim/mars.duckdb` — view: `mars_pairs`

**Next:** `03b_srs_scores_mars.ipynb` — compute SRS per session and attach to pairs